In [1]:
print("hello")

hello


In [2]:
# !pip install langchain-pinecone pinecone-client

In [3]:
# !pip install -U langchain-pinecone

In [4]:
# !pip install -qU langchain-pinecone

In [5]:
# pip install -qU langchain langchain-pinecone 

In [6]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone, ServerlessSpec
load_dotenv()

True

In [7]:
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
print(str(PINECONE_API_KEY))

pcsk_3snxbU_2qN4QnURwsXFcvg8YAaAJos2nwkAyEu9oiAXnmvbnopn7FMTr3kE1smeQnppN2R


In [8]:
pc = Pinecone(api_key = str(PINECONE_API_KEY))

In [9]:
index_name = 'cone'
if not pc.has_index(index_name):
    pc.create_index(
        name = 'cone',
        dimension = 1024,
        metric = 'cosine',
        spec = ServerlessSpec(cloud = 'aws', region = 'us-east-1')
    )

index = pc.Index(index_name)

In [10]:
index

In [11]:
from langchain_pinecone import PineconeEmbeddings
embedding_model = PineconeEmbeddings(model = 'multilingual-e5-large')

In [12]:
from langchain_pinecone import PineconeVectorStore
vectorstore = PineconeVectorStore(
    index = index, 
    embedding = embedding_model
)

In [13]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
    id=1,
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
    id=2,
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
    id=3,
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
    id=4,
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
    id=5,
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
    id=6,
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
    id=7,
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
    id=8,
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
    id=9,
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
    id=10,
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
uuids = [str(uuid4()) for _ in range(len(documents))]
vectorstore.add_documents(documents=documents, ids=uuids)

['ad0b9e10-1ca3-4ecf-8aab-34161dfdf2f5',
 '2818fbd8-8d65-41ca-a01a-7e9b89e4c858',
 '804aa9b1-2970-4998-881f-72929f7f5e16',
 '4c2c8fb3-354b-48ff-ab1d-dbd8e22f13a1',
 'c87787eb-c61b-4d9a-b0f1-f2499add87e0',
 '8e524de5-d518-4069-8a53-74e3e3b9558d',
 '125eda9e-09c3-458d-983c-6dcc1dc79252',
 'ab8c46c1-e106-4365-a030-47b7db345b08',
 '404de42a-72ac-4c3f-9a79-da9b716c8688',
 'e9698d03-0fc9-4c0f-81c0-6cc9695d7ac1']

In [14]:
retriever = vectorstore.as_retriever(
    search_kwargs = {"k" : 3 }
)

In [15]:
retriever.invoke("Langchain framework for beginner now?")

[Document(id='804aa9b1-2970-4998-881f-72929f7f5e16', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='ab8c46c1-e106-4365-a030-47b7db345b08', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='e9698d03-0fc9-4c0f-81c0-6cc9695d7ac1', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

In [16]:
from langchain_pinecone import PineconeRerank

In [17]:
re_ranker = PineconeRerank()

In [18]:
re_ranker.compress_documents(retriever.invoke("Explain transformer architecture"),"Explain transformer architecture")

[Document(metadata={'source': 'tweet', 'relevance_score': 7.543321e-05}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'tweet', 'relevance_score': 5.6497935e-05}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(metadata={'source': 'tweet', 'relevance_score': 1.6187581e-05}, page_content='I have a bad feeling I am going to get deleted :(')]